In [85]:
import torch
import torch.nn as nn
import torchvision.datasets as datasets
import torchvision.transforms as transforms

In [86]:
input_size = 784
hidden_size = 40
output_size = 10
epochs = 10
batch_size = 100
learning_rate = 0.001

In [87]:
train_dataset = datasets.MNIST(root="./data", train=True, transform=transforms.ToTensor(), download=True)
test_dataset = datasets.MNIST(root="./data", train=False, transform=transforms.ToTensor())

In [88]:
train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

In [89]:
class Net(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, output_size)
        self.init_weights()

    def init_weights(self):
        nn.init.kaiming_normal_(self.fc1.weight)
        nn.init.kaiming_normal_(self.fc2.weight)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.fc3(x)

        return x

In [90]:
net = Net(784, 400, 10)

CUDA = torch.cuda.is_available()

if CUDA:
    net = net.cuda()

optimizer = torch.optim.Adam(params=net.parameters(), lr=learning_rate)
loss_func = torch.nn.CrossEntropyLoss()

for epoch in range(epochs):
    correct_train = 0
    running_loss = 0
    for i, (images, labels) in enumerate(train_loader):
        images = images.view(-1, 28 * 28)

        if CUDA:
            images = images.cuda()
            labels = labels.cuda()

        outputs = net(images)
        (_, predicted) = torch.max(outputs.data, 1)
        correct_train += (labels == predicted).sum()

        loss = loss_func(outputs, labels)
        running_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print("Epoch: {}/{} Training Loss: {:.3f}, Training Accuracy: {:.3f}".format((epoch + 1), epochs, running_loss, (correct_train.double() / len(train_dataset)) * 100))

Epoch: 1/10 Training Loss: 142.884, Training Accuracy: 93.045
Epoch: 2/10 Training Loss: 50.622, Training Accuracy: 97.448
Epoch: 3/10 Training Loss: 34.434, Training Accuracy: 98.122
Epoch: 4/10 Training Loss: 22.209, Training Accuracy: 98.807
Epoch: 5/10 Training Loss: 17.816, Training Accuracy: 99.027
Epoch: 6/10 Training Loss: 13.076, Training Accuracy: 99.300
Epoch: 7/10 Training Loss: 11.242, Training Accuracy: 99.362
Epoch: 8/10 Training Loss: 9.806, Training Accuracy: 99.443
Epoch: 9/10 Training Loss: 9.972, Training Accuracy: 99.435
Epoch: 10/10 Training Loss: 8.374, Training Accuracy: 99.515


In [91]:
with torch.no_grad():
    correct = 0
    for (images, labels) in test_loader:
        images = images.view(-1, 28 * 28)
        if CUDA:
            images = images.cuda()
            labels = labels.cuda()

        outputs = net(images)
        (_, predicted) = torch.max(outputs.data, 1)
        correct += (labels == predicted).sum().item()

    print(correct)
    print("Accuracy of the network over 10000 test image {} %".format(100 *  correct / len(test_dataset)))


9816
Accuracy of the network over 10000 test image 98.16 %
